## Consuming data using Kafka and Visualise (20%)
In this task, we will implement an Apache Kafka consumer to consume the data from Part 2.  
  
Important:   
-	In this part, Kafka consumers are used to consume the streaming data published from task 2.8.

In [ ]:
# Consumer

2. Create a dashboard with 3 plots to visualise the real-time streaming data from part 2:  
a) A basic plot to show the number of high-severity accidents (line/bar chart).  
b) A histogram shows the cumulative distribution of accidents by severity.  
c) A bubble map plot similar to UK Road Traffic Accidents - Crash View. (Just the map; no interactive controllers are required. You can use any library (like folium, seaborn, plotly, etc.)

Note: Ideally, for a dashboard-like user experience, 3 diagrams need to be updated in real-time simultaneously. You may need to do a bit of searching/research to make it work.

In [ ]:
# %pip install folium

In [ ]:
import json
import time
import threading
from datetime import datetime

import pandas as pd
from kafka import KafkaConsumer

import matplotlib.pyplot as plt
import plotly.express as px
import folium

from IPython.display import clear_output, display
from datetime import datetime
from zoneinfo import ZoneInfo



In [ ]:
TOPIC_HIGH_SEVERITY = "a2b_6a_high_severity"
TOPIC_SEVERITY_COUNT = "a2b_6b_severity_count"
TOPIC_DISTRICT_SEVERITY = "a2b_6c_district_severity"

In [ ]:
high_severity_records = []
severity_count_records = []
district_severity_records = []

MAX_RECORDS = 2000

In [ ]:
# Create a Kafka consumer to read data
def consume_kafka():
    consumer = KafkaConsumer(
        TOPIC_HIGH_SEVERITY,
        TOPIC_SEVERITY_COUNT,
        TOPIC_DISTRICT_SEVERITY,
        bootstrap_servers="kafka:9092",
        auto_offset_reset="latest",
        enable_auto_commit=True,
        group_id="accident_dashboard_notebook_consumer",
        value_deserializer=lambda x: json.loads(x.decode("utf-8"))
    )

    print("Kafka consumer started:")

    for msg in consumer:
        topic = msg.topic
        value = msg.value
        # Add the time when this message is received
        value["received_time"] = datetime.now(ZoneInfo("Europe/London")).strftime("%H:%M:%S")
        # Save data to the correct list based on its topic
        if topic == TOPIC_HIGH_SEVERITY:
            high_severity_records.append(value)
            # Keep only the latest records
            if len(high_severity_records) > MAX_RECORDS:
                high_severity_records.pop(0)

        elif topic == TOPIC_SEVERITY_COUNT:
            severity_count_records.append(value)
            if len(severity_count_records) > MAX_RECORDS:
                severity_count_records.pop(0)

        elif topic == TOPIC_DISTRICT_SEVERITY:
            district_severity_records.append(value)
            if len(district_severity_records) > MAX_RECORDS:
                district_severity_records.pop(0)

In [ ]:
# Run Kafka consumer in the background so the dashboard can update at the same time
consumer_thread = threading.Thread(target=consume_kafka, daemon=True)
consumer_thread.start()

In [ ]:
while True:
    # Clear old dashboard output before showing the new one
    clear_output(wait=True)
    # Convert Kafka records into Pandas DataFrames
    high_df = pd.DataFrame(high_severity_records)
    severity_df = pd.DataFrame(severity_count_records)
    print("Last update:", datetime.now(ZoneInfo("Europe/London")).strftime("%H:%M:%S"))

    # Plot A number of high-severity accidents over time
    import matplotlib.pyplot as plt
    import pandas as pd

    high_df = pd.DataFrame(high_severity_records)

    if "received_time" in high_df.columns:
    
        # Convert received time to hour-minute-second format
        high_df["received_time_sec"] = pd.to_datetime(
            high_df["received_time"],
            format="%H:%M:%S",
            errors="coerce"
        ).dt.strftime("%H:%M:%S")
    
        # Count high-severity accidents for each received time
        high_count_df = (
            high_df
            .dropna(subset=["received_time_sec"])
            .groupby("received_time_sec")
            .size()
            .reset_index(name="high_severity_count")
        )
    
        # Draw bar chart for high-severity accident count
        plt.figure(figsize=(10, 4))
        plt.bar(
            high_count_df["received_time_sec"],
            high_count_df["high_severity_count"]
        )
    
        plt.title("A) Number of High-Severity Accidents Over Time")
        plt.xlabel("Received Time")
        plt.ylabel("Number of High-Severity Accidents")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


    else:
        print("Waiting for high severity data")


    # Plot B: cumulative distribution by severity
    severity_df = pd.DataFrame(severity_count_records)
    
    if {"severity_rating", "count"}.issubset(severity_df.columns):
    
        # Convert columns into numeric values
        severity_df["severity_rating"] = pd.to_numeric(
            severity_df["severity_rating"], errors="coerce"
        )
        severity_df["count"] = pd.to_numeric(
            severity_df["count"], errors="coerce"
        )
    
        # Remove invalid rows
        severity_df = severity_df.dropna(subset=["severity_rating", "count"])
    
        # Use the latest count for each severity rating
        latest_severity_df = (
            severity_df
            .drop_duplicates(subset=["severity_rating"], keep="last")
            .sort_values("severity_rating")
        )
    
        # Draw bar chart for severity distribution
        plt.figure(figsize=(8, 4))
        plt.bar(
            latest_severity_df["severity_rating"],
            latest_severity_df["count"]
        )
    
        plt.title("B) Cumulative Distribution of Accidents by Severity")
        plt.xlabel("Severity Rating")
        plt.ylabel("Total Number of Accidents")
        plt.xticks(range(1, 11))
        plt.tight_layout()
        plt.show()
    
    else:
        print("Waiting for severity count data")

    # Plot C bubble map 
    if {"latitude", "longitude","severity_rating"}.issubset(high_df.columns):
        map_df = high_df.copy()
        # Convert map columns into numeric values
        map_df["latitude"] = pd.to_numeric(map_df["latitude"], errors="coerce")
        map_df["longitude"] = pd.to_numeric(map_df["longitude"], errors="coerce")
        map_df["severity_rating"] = pd.to_numeric(map_df["severity_rating"], errors="coerce")
        # Remove rows without valid location or severity
        map_df = map_df.dropna(subset=["latitude", "longitude", "severity_rating"])
    
        if not map_df.empty:
            # Set the map center based on accident locations
            center_lat = map_df["latitude"].mean()
            center_lon = map_df["longitude"].mean()
            # Create the map
            accident_map = folium.Map(
                location=[center_lat, center_lon],
                zoom_start=10,
                tiles="OpenStreetMap"
            )
            # Add one bubble for each high-severity accident
            for _, row in map_df.iterrows():
                radius = float(row["severity_rating"]) * 2
    
                folium.CircleMarker(
                    location=[row["latitude"], row["longitude"]],
                    radius=radius,
                    fill=True,
                    fill_opacity=0.6,
                    color="blue",
                    fill_color="blue"
                ).add_to(accident_map)
            
            print("C) Bubble Map of High-Severity Accidents")
            display(accident_map)
    
        else:
            print("Waiting for valid latitude and longitude data")
    else:
        print("Waiting for map data")

    time.sleep(5)